In [1]:
import jax
import jax.numpy as jnp
import numpy as np

In [2]:
# !pip install treescope
import treescope

treescope.basic_interactive_setup(autovisualize_arrays=True)

In [3]:
jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0),
 TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0),
 TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0),
 TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0),
 TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

In [4]:
# !pip install -U xprof
# !pip install -U protobuf

In [5]:
with jax.profiler.trace("/tmp/tensorboard"):
    key = jax.random.key(0)
    x = jax.random.normal(key, (32, 1024, 8192)).astype(jnp.bfloat16)
    W = jax.random.normal(key, (8192, 8192 * 4)).astype(jnp.bfloat16)
    y = x @ W
    y.block_until_ready()

2026-07-11 06:28:31.882322: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-11 06:28:44.486704: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [6]:
%load_ext tensorboard

In [8]:
%tensorboard --logdir=/tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 15562), started 0:00:05 ago. (Use '!kill 15562' to kill it.)

# Play around with stuff!

In [9]:
import jax
import jax.numpy as jnp
import treescope

treescope.basic_interactive_setup(autovisualize_arrays=True)

from jax.sharding import PartitionSpec as P

# Can also do this to "mock" 1024 devices for the sake of parallelism.
# jax.config.update('jax_num_cpu_devices', 1024)

In [10]:
def forward(
    x: jax.Array,  # [B, T],
    embed: jax.Array,  # [V, D],
    w1: jax.Array,  # [D, F],
    w2: jax.Array,  # [F, D]
) -> jax.Array:

    # Embed input tokens [B, T] -> [B, T, D]
    x = embed[x, :]

    # [B, T, D] -> [B, T, F] -> [B, T, D]
    x = jnp.dot(jax.nn.relu(jnp.dot(x, w1)), w2)

    # [B, T, D] -> [B, T, V]
    return jax.nn.log_softmax(jnp.dot(x, embed.T))

In [11]:
bs = 8
seqlen = 1024
vocab_size = 32_768
d_model = 8_192
d_ff = 16_384

tokens = jnp.zeros((bs, seqlen), dtype=jnp.int32)  # tokens of shape (batch_size, seq_len)
embed = jnp.zeros(
    (vocab_size, d_model), jnp.bfloat16
)  # a single weight matrix of shape (d_model, 2 * d_model)
w1 = jnp.zeros(
    (d_model, d_ff), jnp.bfloat16
)  # a single weight matrix of shape (d_model, 2 * d_model)
w2 = jnp.zeros(
    (d_ff, d_model), jnp.bfloat16
)  # a single weight matrix of shape (d_model, 2 * d_model)

forward_fn = jax.jit(forward)

out = forward_fn(tokens, embed, w1, w2)  # this compiles and runs the program

In [12]:
with jax.profiler.trace("/tmp/tensorboard"):
    out = forward_fn(tokens, embed, w1, w2)
    _ = jax.block_until_ready(out)

2026-07-11 06:29:20.030533: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-11 06:29:20.160368: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [13]:
def update_step(x, y, embed, w1, w2, lr):
    def loss_fn(x, y, embed, w1, w2):
        logits = forward_fn(x, embed, w1, w2)
        return -jnp.mean(jnp.take_along_axis(logits, y[:, :, None], axis=-1))

    grad = jax.grad(loss_fn, argnums=(2, 3, 4))(x, y, embed, w1, w2)
    return jax.tree.map(lambda w, grad: w - lr * grad, (embed, w1, w2), grad)

In [14]:
x = tokens
y = jnp.roll(tokens, axis=1, shift=1)

In [15]:
new_embed, new_w1, new_w2 = jax.jit(update_step)(x, y, embed, w1, w2, jnp.asarray(1e-3))

# Run on multiple TPUs

In [16]:
# Create a mesh with a data dimension of size 64, model of 1.
mesh = jax.make_mesh(axis_shapes=(64, 1), axis_names=("data", "model"))
jax.sharding.set_mesh(mesh)

ValueError: Number of devices 8 must be >= the product of mesh_shape (64, 1)

In [ ]:
tokens = jax.device_put(tokens, P("data", None))  # sharded along the batch dimension
embed, w1, w2 = jax.tree.map(
    lambda arr: jax.device_put(arr, None), (embed, w1, w2)
)  # None means "replicated"

In [ ]:
tokens

Array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int32)

In [ ]:
with profile(devices=jax.devices()[0:1]) as url:
    out = forward_fn(tokens, embed, w1, w2)
    _ = jax.block_until_ready(out)

In [ ]:
print(url)

['http://xprof/trace_viewer/jaaustin-1750692262060980414']


# What if we want to do something more complicated?

In [ ]:
# Create a mesh with a data dimension of size 64, model of 1.
mesh = jax.make_mesh(axis_shapes=(64, 1), axis_names=("data", "model"))
jax.sharding.set_mesh(mesh)

ValueError: Number of devices 1 must be >= the product of mesh_shape (64, 1)

In [ ]:
# What about FSDP?

tokens = jax.device_put(tokens, P("data", None))  # sharded along the batch dimension

w1 = jax.device_put(w1, P("data", None))
w2 = jax.device_put(w2, P(None, "data"))
embed = jax.device_put(embed, P(None, None))

NameError: name 'tokens' is not defined

In [ ]:
with profile(devices=jax.devices()[0:1]) as url:
    out = forward_fn(tokens, embed, w1, w2)
    _ = jax.block_until_ready(out)

In [ ]:
print(url[0])

http://xprof/trace_viewer/jaaustin-1750694281044602208


# Model parallelism

In [ ]:
# Create a mesh with a data dimension of size 64, model of 1.
mesh = jax.make_mesh(axis_shapes=(8, 8), axis_names=("data", "model"))
jax.sharding.set_mesh(mesh)

Mesh(device_ids=array([[ 0],
       [ 8],
       [16],
       [24],
       [32],
       [40],
       [48],
       [56],
       [ 1],
       [ 9],
       [17],
       [25],
       [33],
       [41],
       [49],
       [57],
       [ 2],
       [10],
       [18],
       [26],
       [34],
       [42],
       [50],
       [58],
       [ 3],
       [11],
       [19],
       [27],
       [35],
       [43],
       [51],
       [59],
       [ 4],
       [12],
       [20],
       [28],
       [36],
       [44],
       [52],
       [60],
       [ 5],
       [13],
       [21],
       [29],
       [37],
       [45],
       [53],
       [61],
       [ 6],
       [14],
       [22],
       [30],
       [38],
       [46],
       [54],
       [62],
       [ 7],
       [15],
       [23],
       [31],
       [39],
       [47],
       [55],
       [63]]), axis_names=('data', 'model'), axis_types=(Auto, Auto))

In [ ]:
tokens = jax.device_put(tokens, P("data", None))  # sharded along the batch dimension

w1 = jax.device_put(w1, P(None, "model"))
w2 = jax.device_put(w2, P("model", None))
embed = jax.device_put(embed, P(None, "model"))

In [ ]:
with profile(devices=jax.devices()[0:1]) as url:
    out = forward_fn(tokens, embed, w1, w2)
    _ = jax.block_until_ready(out)

In [ ]:
print(url[0])

http://xprof/trace_viewer/jaaustin-1750693862952331409


# Writing a "clean" LLM in JAX

In [18]:
from typing import Callable

from flax import struct


@struct.dataclass
class TensorInfo:
    shape: jax.ShapeDtypeStruct
    logical_axes: tuple[str, ...]
    initializer: Callable | None = None


@struct.dataclass
class Config:
    num_layers: int
    d_model: int
    d_ff: int
    vocab_size: int

    dtype: jnp.dtype = jnp.bfloat16


@struct.dataclass
class Layer:
    w1: jax.Array | TensorInfo
    w2: jax.Array | TensorInfo

    @classmethod
    def abstract(cls, cfg: Config):
        return Layer(
            w1=TensorInfo(
                jax.ShapeDtypeStruct((cfg.d_model, cfg.d_ff), cfg.dtype),
                ("d_model", "d_ff"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=1),
            ),
            w2=TensorInfo(
                jax.ShapeDtypeStruct((cfg.d_ff, cfg.d_model), cfg.dtype),
                ("d_ff", "d_model"),
                jax.nn.initializers.he_normal(in_axis=1, out_axis=0),
            ),
        )


@struct.dataclass
class Weights:
    embedding: jax.Array | TensorInfo
    layers: list[Layer]

    @classmethod
    def abstract(cls, cfg: Config):
        return Weights(
            layers=[Layer.abstract(cfg) for _ in range(cfg.num_layers)],
            embedding=TensorInfo(
                jax.ShapeDtypeStruct((cfg.vocab_size, cfg.d_model), cfg.dtype),
                ("vocab", "d_model"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=1),
            ),
        )

    @classmethod
    def shardings(cls, cfg: Config, mesh: jax.sharding.Mesh, rules: dict):
        abstract = cls.abstract(cfg)
        return jax.tree.map(
            lambda info: _logical_to_sharding(info.logical_axes, mesh, rules),
            abstract,
            is_leaf=lambda x: isinstance(x, TensorInfo),
        )

    @classmethod
    def init(cls, cfg: Config, key: jax.random.PRNGKey, mesh: jax.sharding.Mesh, rules: dict):
        def _init():
            abstract = cls.abstract(cfg)
            # Create one new RNG key per tensor.
            num_leaves = len(jax.tree_util.tree_leaves(abstract))
            key_iter = iter(jax.random.split(key, num_leaves))
            return jax.tree.map(
                lambda info: info.initializer(next(key_iter), info.shape.shape, info.shape.dtype),
                abstract,
                is_leaf=lambda x: isinstance(x, TensorInfo),
            )

        _init = jax.jit(_init, out_shardings=cls.shardings(cfg, mesh, rules))
        return jax.device_put(_init(), cls.shardings(cfg, mesh, rules))

In [19]:
import collections
from typing import Tuple

ShardingRules = collections.namedtuple(
    "ShardingRules", ["batch", "sequence", "d_model", "d_ff", "vocab"]
)


def _logical_to_physical(logical: Tuple[str, ...], rules: ShardingRules):
    """Converts logical to physical pspec."""
    return P(*(getattr(rules, l) if l is not None else None for l in logical))


def _logical_to_sharding(logical: Tuple[str, ...], mesh: jax.sharding.Mesh, rules: ShardingRules):
    """Converts logical to sharding."""
    return jax.sharding.NamedSharding(mesh, _logical_to_physical(logical, rules))


fsdp_rules = ShardingRules(batch="x", sequence=None, d_model="x", d_ff=None, vocab=None)

mdl_parallel_rules = ShardingRules(batch=None, sequence=None, d_model=None, d_ff="x", vocab=None)


def _logical_to_physical(logical: P, rules: ShardingRules):
    """Converts logical to physical pspec."""
    return P(*(getattr(rules, l) for l in logical))


def _logical_to_sharding(logical: P, mesh: jax.sharding.Mesh, rules: ShardingRules):
    """Converts logical to sharding."""
    return jax.sharding.NamedSharding(mesh, _logical_to_physical(logical, rules))

In [20]:
jax.sharding.set_mesh(None)

In [21]:
cfg = Config(
    d_model=8192,
    d_ff=8192 * 4,
    num_layers=4,
    vocab_size=8192,
    dtype=jnp.bfloat16,
)

mesh = jax.make_mesh((8, 8), ("x", "y"))

rules = fsdp_rules

rng = jax.random.PRNGKey(42)
weights = Weights.init(cfg, rng, mesh, rules)

ValueError: Number of devices 8 must be >= the product of mesh_shape (8, 8)

In [22]:
def forward(x: jax.Array, weights: Weights):

    # Embed input tokens [B, T] -> [B, T, D]
    x = weights.embedding[x, :]

    for layer in weights.layers:
        # [B, T, D] -> [B, T, D] -> [B, T, D]
        x = jax.nn.relu(jnp.dot(x, layer.w1))
        x = jax.nn.relu(jnp.dot(x, layer.w2))

    # [B, T, D] -> [B, T, V]
    return jax.nn.log_softmax(jnp.dot(x, weights.embedding.T))

In [23]:
batch_size = 32
seq_len = 1024

input_sharding = _logical_to_sharding(("batch", "sequence"), mesh=mesh, rules=rules)
x = jnp.zeros((batch_size, seq_len), dtype=jnp.int32, device=input_sharding)

output_sharding = _logical_to_sharding(("batch", "sequence", "vocab"), mesh=mesh, rules=rules)
forward_fn = jax.jit(forward, out_shardings=output_sharding)

compiled = forward_fn.lower(x, weights).compile()

NameError: name 'mesh' is not defined

# TLDR

In [24]:
# Data parallelism
activations = jax.device_put(
    activations, P("data", None, None)
)  # sharded along the batch dimension
weights = jax.device_put(weights, P(None, None))  # unsharded

# FSDP parallelism
activations = jax.device_put(
    activations, P("data", None, None)
)  # sharded along the batch dimension
weights = jax.device_put(weights, P("data", None))  # weights sharded and gathered just-in-time

# Tensor parallelism
activations = jax.device_put(
    activations, P("data", None, "model")
)  # sharded along the batch and hidden
weights = jax.device_put(weights, P(None, "model"))  # weights sharded and gathered just-in-time

# Mixed FSDP + model parallelism
activations = jax.device_put(
    activations, P("data", None, "model")
)  # sharded along the batch and hidden
weights = jax.device_put(weights, P("data", "model"))  # we do both! fully sharded weights

NameError: name 'activations' is not defined

# Sharding in Types

In [25]:
from jax.sharding import AxisType
from jax.sharding import PartitionSpec as P

ImportError: cannot import name 'reshard' from 'jax.sharding' (/home/hyeonseok/.local/lib/python3.10/site-packages/jax/sharding.py)

In [26]:
# Create a mesh with a data dimension of size 64, model of 1.
mesh = jax.make_mesh(
    axis_shapes=(8, 8),
    axis_names=("data", "model"),
    axis_types=(AxisType.Explicit, AxisType.Explicit),
)
jax.sharding.set_mesh(mesh)

ValueError: Number of devices 8 must be >= the product of mesh_shape (8, 8)

In [ ]:
tokens = jnp.zeros((128, 1024), dtype=jnp.bfloat16)  # tokens of shape (batch_size, seq_len)
weights = jnp.zeros(
    (1024, 2048), jnp.bfloat16
)  # a single weight matrix of shape (d_model, 2 * d_model)

# data parallelism
tokens = jax.device_put(tokens, P("data", None))  # sharded along the batch dimension
weights = jax.device_put(weights, P("data", "model"))  # unsharded


def update_step(tokens, weights):
    print(jax.typeof(tokens).sharding)
    out = jnp.matmul(tokens, weights)
    print(jax.typeof(out).sharding)
    return out


update_fn = jax.jit(update_step)  # some training step function, using jax.grad

new_weights = update_fn(tokens, weights)  # this compiles and runs the program

NamedSharding(mesh=AbstractMesh('data': 8, 'model': 8, axis_types=(Explicit, Explicit)), spec=PartitionSpec('data', None))
NamedSharding(mesh=AbstractMesh('data': 8, 'model': 8, axis_types=(Explicit, Explicit)), spec=PartitionSpec('data', 'model'))


# Shard Map

In [ ]:
import functools

import jax
import jax.numpy as jnp
import jax.sharding as shd
from jax.experimental.shard_map import shard_map

mesh = jax.make_mesh(axis_shapes=(8, 8), axis_names=("X", "Y"))


def P(*args):
    return shd.NamedSharding(mesh, shd.PartitionSpec(*args))


B, D, F = 1024, 2048, 8192
A = jnp.arange(np.prod((B, D))).reshape((B, D))
W = jnp.arange(np.prod((D, F))).reshape((D, F))

A = jax.device_put(A, P("X", "Y"))
W = jax.device_put(W, P(None, "Y"))


@functools.partial(jax.jit, out_shardings=P("X", "Y"))
def matmul(lhs, rhs):
    return lhs @ rhs


def collective_matmul_allgather_lhs_contracting(lhs, rhs):
    # lhs is the looped operand; rhs is the local operand
    axis_size = jax.lax.psum(1, axis_name="Y")  # axis_size = 4 for this example
    idx = jax.lax.axis_index("Y")

    chunk_size = lhs.shape[1]
    assert rhs.shape[0] % chunk_size == 0

    def f(i, carrys):
        accum, lhs = carrys
        rhs_chunk = jax.lax.dynamic_slice_in_dim(
            rhs, (idx + i) % axis_size * chunk_size, chunk_size
        )
        # Matmul for a chunk
        update = lhs @ rhs_chunk
        # Circular shift to the left
        lhs = jax.lax.ppermute(
            lhs, axis_name="Y", perm=[(j, (j - 1) % axis_size) for j in range(axis_size)]
        )
        return accum + update, lhs

    accum = jnp.zeros((lhs.shape[0], rhs.shape[1]), dtype=lhs.dtype)
    accum, lhs = jax.lax.fori_loop(0, axis_size - 1, f, (accum, lhs), unroll=True)

    # Compute the last chunk after the final permute to leave lhs in the state we found it
    i = axis_size - 1
    rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, (idx + i) % axis_size * chunk_size, chunk_size)
    update = lhs @ rhs_chunk
    return accum + update


jit_sharded_f = jax.jit(
    shard_map(
        collective_matmul_allgather_lhs_contracting,
        mesh,
        in_specs=(shd.PartitionSpec("X", "Y"), shd.PartitionSpec(None, "Y")),
        out_specs=shd.PartitionSpec("X", "Y"),
    )
)

shmapped_out = jit_sharded_f(A, W)
expected_out = matmul(A, W)

np.testing.assert_array_equal(shmapped_out, expected_out)

INFO:2025-06-23 10:00:01,725:jax._src.xla_bridge:755: Unable to initialize backend 'tpu': UNKNOWN: TPU initialization failed: No jellyfish device found.
INFO:2025-06-23 10:00:01,727:jax._src.xla_bridge:755: Unable to initialize backend 'pathways': Could not initialize backend 'pathways'
INFO:2025-06-23 10:00:01,730:jax._src.xla_bridge:755: Unable to initialize backend 'proxy': INVALID_ARGUMENT: IFRT proxy server address must be '<transport-type>://<backend-address>' (e.g., 'grpc://localhost'), but got 
INFO:2025-06-23 10:00:01,732:jax._src.xla_bridge:755: Unable to initialize backend 'mlcr': Could not initialize backend 'mlcr'
INFO:2025-06-23 10:00:01,733:jax._src.xla_bridge:755: Unable to initialize backend 'sliceme': Could not initialize backend 'sliceme'
INFO:2025-06-23 10:00:01,734:jax._src.xla_bridge:755: Unable to initialize backend 'mock_tpu': Must pass --mock_tpu_platform flag to initialize the mock_tpu backend


ValueError: Number of devices 1 must be >= the product of mesh_shape (8, 8)

## Mini Transformer

In [27]:
import collections
from typing import Callable, Tuple

import jax
import jax.numpy as jnp
from flax import struct

P = jax.sharding.PartitionSpec

ShardingRules = collections.namedtuple(
    "ShardingRules",
    ["batch", "sequence", "d_model", "query_heads", "key_heads", "key_dim", "d_ff", "vocab"],
)


def _logical_to_physical(logical: Tuple[str, ...], rules: ShardingRules):
    """Converts logical to physical pspec."""
    return P(*(getattr(rules, l) for l in logical))


def _logical_to_sharding(logical: Tuple[str, ...], mesh: jax.sharding.Mesh, rules: ShardingRules):
    """Converts logical to sharding."""
    return jax.sharding.NamedSharding(mesh, _logical_to_physical(logical, rules))


@struct.dataclass
class Config:
    d_model: int
    ffw_multiplier: int
    num_layers: int

    query_heads: int
    kv_heads: int
    key_dim: int

    vocab_size: int

    dtype: jnp.dtype = jnp.bfloat16


@struct.dataclass
class TensorInfo:
    shape: jax.ShapeDtypeStruct
    logical_axes: tuple[str, ...]
    initializer: Callable | None = None


@struct.dataclass
class Layer:
    q: jax.Array | TensorInfo
    k: jax.Array | TensorInfo
    v: jax.Array | TensorInfo
    proj: jax.Array | TensorInfo

    w1: jax.Array | TensorInfo
    w2: jax.Array | TensorInfo

    gamma1: jax.Array | TensorInfo
    gamma2: jax.Array | TensorInfo

    @classmethod
    def abstract(cls, cfg: Config):
        return Layer(
            q=TensorInfo(
                jax.ShapeDtypeStruct((cfg.d_model, cfg.query_heads, cfg.key_dim), dtype=cfg.dtype),
                ("d_model", "query_heads", "key_dim"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=(1, 2)),
            ),
            k=TensorInfo(
                jax.ShapeDtypeStruct((cfg.d_model, cfg.kv_heads, cfg.key_dim), dtype=cfg.dtype),
                ("d_model", "query_heads", "key_dim"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=(1, 2)),
            ),
            v=TensorInfo(
                jax.ShapeDtypeStruct((cfg.d_model, cfg.kv_heads, cfg.key_dim), dtype=cfg.dtype),
                ("d_model", "query_heads", "key_dim"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=(1, 2)),
            ),
            proj=TensorInfo(
                jax.ShapeDtypeStruct((cfg.query_heads, cfg.key_dim, cfg.d_model), dtype=cfg.dtype),
                ("query_heads", "key_dim", "d_model"),
                jax.nn.initializers.he_normal(in_axis=(0, 1), out_axis=2),
            ),
            w1=TensorInfo(
                jax.ShapeDtypeStruct(
                    (cfg.d_model, cfg.d_model * cfg.ffw_multiplier), dtype=cfg.dtype
                ),
                ("d_model", "d_ff"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=1),
            ),
            w2=TensorInfo(
                jax.ShapeDtypeStruct(
                    (cfg.d_model * cfg.ffw_multiplier, cfg.d_model), dtype=cfg.dtype
                ),
                ("d_ff", "d_model"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=1),
            ),
            gamma1=TensorInfo(
                jax.ShapeDtypeStruct((cfg.d_model,), dtype=cfg.dtype),
                ("d_model",),
                jax.nn.initializers.constant(1.0),
            ),
            gamma2=TensorInfo(
                jax.ShapeDtypeStruct((cfg.d_model,), dtype=cfg.dtype),
                ("d_model",),
                jax.nn.initializers.constant(1.0),
            ),
        )


@struct.dataclass
class Weights:
    layers: list[Layer]
    embedding: jax.Array | TensorInfo

    @classmethod
    def abstract(cls, cfg: Config):
        return Weights(
            layers=[Layer.abstract(cfg) for _ in range(cfg.num_layers)],
            embedding=TensorInfo(
                jax.ShapeDtypeStruct((cfg.vocab_size, cfg.d_model), cfg.dtype),
                ("vocab", "d_model"),
                jax.nn.initializers.he_normal(in_axis=0, out_axis=1),
            ),
        )

    @classmethod
    def sharding(cls, cfg: Config, mesh: jax.sharding.Mesh, rules: ShardingRules):
        abstract = cls.abstract(cfg)
        return jax.tree.map(
            lambda info: _logical_to_sharding(info.logical_axes, mesh, rules),
            abstract,
            is_leaf=lambda x: isinstance(x, TensorInfo),
        )

    @classmethod
    def init(
        cls, cfg: Config, key: jax.random.PRNGKey, mesh: jax.sharding.Mesh, rules: ShardingRules
    ):
        abstract = cls.abstract(cfg)

        def _init():
            num_leaves = len(jax.tree.leaves(abstract, is_leaf=lambda x: isinstance(x, TensorInfo)))
            rng_iter = iter(jax.random.split(key, num_leaves))
            return jax.tree.map(
                lambda info: info.initializer(next(rng_iter), info.shape.shape, info.shape.dtype),
                abstract,
                is_leaf=lambda x: isinstance(x, TensorInfo),
            )

        sharding = cls.sharding(cfg, mesh, rules)
        return jax.jit(_init, out_shardings=sharding)()

In [57]:
def rms_norm(x: jax.Array, gamma: jax.Array) -> jax.Array:
    """Apply RMS normalization."""
    rms = jax.lax.rsqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + 1e-6)
    return gamma * x * rms


def layer_forward(x: jax.Array, layer: Layer) -> jax.Array:
    x = jax.lax.with_sharding_constraint(x, jax.NamedSharding(mesh, P("x", None, "y")))

    with jax.named_scope("pre_attn_norm"):
        attn_in = rms_norm(x, layer.gamma1)

    # Q, K, V projections
    with jax.named_scope("qkv_matmul"):
        q = jnp.einsum("btd,dhq->bhtq", attn_in, layer.q)
        k = jnp.einsum("btd,dhk->bhtk", attn_in, layer.k)
        v = jnp.einsum("btd,dhv->bhtv", attn_in, layer.v)

    # TODO: add RoPE here

    # Attention
    with jax.named_scope("attn"):
        scale = q.shape[-1] ** -0.5
        # TODO: add masking here

        num_query_heads, num_kv_heads = q.shape[1], k.shape[1]

        if num_query_heads == num_kv_heads or num_kv_heads == 1:
            qk = jnp.einsum("bhtd,bhsd->bhts", q, k) * scale
            logits = jax.nn.softmax(qk.astype(jnp.float32), axis=-1)
            attn_vec = jnp.einsum("bhsd,bhts->bhtd", v, logits)
        else:
            assert num_query_heads % num_kv_heads == 0
            q = q.reshape(
                q.shape[0:1] + (num_kv_heads, num_query_heads // num_kv_heads) + q.shape[2:]
            )
            qk = jnp.einsum("bqhtd,bhsd->bqhts", q, k) * scale
            logits = jax.nn.softmax(qk.astype(jnp.float32), axis=-1)
            attn_vec = jnp.einsum("bqsd,bqhts->bqhtd", v, logits)
            attn_vec = attn_vec.reshape(
                attn_vec.shape[0:1] + (num_query_heads,) + attn_vec.shape[3:]
            )

    # Attention proj
    with jax.named_scope("attn_proj"):
        attn_out = jnp.einsum("bhtv,hvd->btd", attn_vec, layer.proj)

    # Residual connection
    with jax.named_scope("residual"):
        x = x + attn_out

    # Second RMSNorm (Pre-LN for FFN)
    with jax.named_scope("ffn_pre_norm"):
        ffw_in = rms_norm(x, layer.gamma2)

    ffw_in = jax.lax.with_sharding_constraint(ffw_in, jax.NamedSharding(mesh, P("x", None, None)))
    with jax.named_scope("ffw"):
        c_layer_w1 = jax.lax.with_sharding_constraint(
            layer.w1, jax.NamedSharding(mesh, P(None, "y"))
        )
        ffw_out = jnp.einsum("btd,df->btf", ffw_in, c_layer_w1).astype(jnp.bfloat16)
        ffw_out = jax.nn.gelu(ffw_out)
        c_layer_w2 = jax.lax.with_sharding_constraint(
            layer.w2, jax.NamedSharding(mesh, P("y", None))
        )
        ffw_out = jnp.einsum("btf,fd->btd", ffw_out, c_layer_w2).astype(jnp.bfloat16)

    ffw_out = jax.lax.with_sharding_constraint(ffw_out, jax.NamedSharding(mesh, P("x", None, "y")))
    # Residual connection
    with jax.named_scope("residual"):
        x = x + ffw_out

    return x


def forward(x: jax.Array, weights: Weights) -> jax.Array:
    """Forward pass through the network."""

    one_hot = jax.nn.one_hot(x, cfg.vocab_size)
    x = jnp.einsum("vd,btv->btd", weights.embedding, one_hot)

    for idx, layer in enumerate(weights.layers):
        with jax.named_scope(f"layer_{idx}"):
            x = layer_forward(x, layer)

    logits = jnp.einsum("vd,btd->btv", weights.embedding, x)
    return jax.nn.log_softmax(logits, axis=-1)

In [58]:
fsdp_rules = ShardingRules(
    batch=("x", "y"),
    sequence=None,
    d_model=("x", "y"),
    query_heads=None,
    key_heads=None,
    key_dim=None,
    d_ff=None,
    vocab=None,
)

model_parallel_rules = ShardingRules(
    batch=None,
    sequence=None,
    d_model=None,
    query_heads=("x", "y"),
    key_heads=("x", "y"),
    key_dim=None,
    d_ff=("x", "y"),
    vocab=None,
)

mixed_rules = ShardingRules(
    batch="x",
    sequence=None,
    d_model="x",
    query_heads="y",
    key_heads="y",
    key_dim=None,
    d_ff="y",
    vocab=None,
)

In [59]:
for arr in jax.live_arrays():
    arr.delete()

In [60]:
cfg = Config(
    d_model=8192,
    ffw_multiplier=4,
    num_layers=8,
    query_heads=16,
    kv_heads=16,
    key_dim=256,
    vocab_size=32_128,
    dtype=jnp.bfloat16,
)

mesh = jax.make_mesh((4, 2), ("x", "y"))

rules = mixed_rules

rng = jax.random.PRNGKey(42)
weights = Weights.init(cfg, rng, mesh, rules)

In [61]:
batch_size = 8
seq_len = 1024

input_sharding = _logical_to_sharding(("batch", "sequence"), mesh=mesh, rules=rules)
x = jnp.zeros((batch_size, seq_len), dtype=jnp.int32, device=input_sharding)

compiled = jax.jit(forward).lower(x, weights).compile()

In [62]:
with jax.profiler.trace("/tmp/tensorboard"):
    logits = compiled(x, weights)
    jax.block_until_ready(logits)

2026-07-11 07:25:59.583967: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-11 07:25:59.867384: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [63]:
%tensorboard --logdir=/tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 15562), started 0:58:23 ago. (Use '!kill 15562' to kill it.)

## What's wrong with this trace?

What's wrong with this trace? Try to identify how long each part should take and how long it's actually taking.

Can you fix it? Hint: try adding sharding constraints!

Answer:

In [ ]:
def rms_norm(x: jax.Array, gamma: jax.Array) -> jax.Array:
    """Apply RMS normalization."""
    rms = jax.lax.rsqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + 1e-6)
    return gamma * x * rms


def layer_forward(x: jax.Array, layer: Layer) -> jax.Array:

    x = jax.lax.with_sharding_constraint(x, jax.NamedSharding(mesh, P("x", None, "y")))

    with jax.named_scope("pre_attn_norm"):
        attn_in = rms_norm(x, layer.gamma1)

    # Q, K, V projections
    with jax.named_scope("qkv_matmul"):
        q = jnp.einsum("btd,dhq->bhtq", attn_in, layer.q)
        k = jnp.einsum("btd,dhk->bhtk", attn_in, layer.k)
        v = jnp.einsum("btd,dhv->bhtv", attn_in, layer.v)

    # TODO: add RoPE here

    # Attention
    with jax.named_scope("attn"):
        scale = q.shape[-1] ** -0.5
        # TODO: add masking here

        num_query_heads, num_kv_heads = q.shape[1], k.shape[1]

        if num_query_heads == num_kv_heads or num_kv_heads == 1:
            qk = jnp.einsum("bhtd,bhsd->bhts", q, k) * scale
            logits = jax.nn.softmax(qk.astype(jnp.float32), axis=-1)
            attn_vec = jnp.einsum("bhsd,bhts->bhtd", v, logits)
        else:
            assert num_query_heads % num_kv_heads == 0
            q = q.reshape(
                q.shape[0:1] + (num_kv_heads, num_query_heads // num_kv_heads) + q.shape[2:]
            )
            qk = jnp.einsum("bqhtd,bhsd->bqhts", q, k) * scale
            logits = jax.nn.softmax(qk.astype(jnp.float32), axis=-1)
            attn_vec = jnp.einsum("bqsd,bqhts->bqhtd", v, logits)
            attn_vec = attn_vec.reshape(
                attn_vec.shape[0:1] + (num_query_heads,) + attn_vec.shape[3:]
            )

    # Attention proj
    with jax.named_scope("attn_proj"):
        attn_out = jnp.einsum("bhtv,hvd->btd", attn_vec, layer.proj)

    # Residual connection
    with jax.named_scope("residual"):
        x = x + attn_out

    # Second RMSNorm (Pre-LN for FFN)
    with jax.named_scope("ffn_pre_norm"):
        ffw_in = rms_norm(x, layer.gamma2)

    ffw_in = jax.lax.with_sharding_constraint(ffw_in, jax.NamedSharding(mesh, P("x", None, "y")))

    with jax.named_scope("ffw"):
        w1, w2 = jax.lax.with_sharding_constraint(
            (layer.w1, layer.w2),
            (jax.NamedSharding(mesh, P(None, "y")), jax.NamedSharding(mesh, P("y", None))),
        )

        ffw_out = jnp.einsum("btd,df->btf", ffw_in, w1).astype(jnp.bfloat16)

        ffw_out = jax.lax.with_sharding_constraint(
            ffw_out, jax.NamedSharding(mesh, P("x", None, "y"))
        )

        ffw_out = jax.nn.gelu(ffw_out)
        ffw_out = jnp.einsum("btf,fd->btd", ffw_out, w2).astype(jnp.bfloat16)

    # Residual connection
    with jax.named_scope("residual"):
        x = x + ffw_out

    x = jax.lax.with_sharding_constraint(x, jax.NamedSharding(mesh, P("x", None, "y")))

    return x


def forward(x: jax.Array, weights: Weights) -> jax.Array:
    """Forward pass through the network."""

    one_hot = jax.nn.one_hot(x, cfg.vocab_size)
    x = jnp.einsum("vd,btv->btd", weights.embedding, one_hot)

    for idx, layer in enumerate(weights.layers):
        with jax.named_scope(f"layer_{idx}"):
            x = layer_forward(x, layer)

    logits = jnp.einsum("vd,btd->btv", weights.embedding, x)
    return jax.nn.log_softmax(logits, axis=-1)

In [ ]:
for arr in jax.live_arrays():
    arr.delete()

In [ ]:
cfg = Config(
    d_model=8192,
    ffw_multiplier=4,
    num_layers=4,
    query_heads=16,
    kv_heads=16,
    key_dim=256,
    vocab_size=32_128,
    dtype=jnp.bfloat16,
)

mesh = jax.make_mesh((4, 2), ("x", "y"))

rules = mixed_rules

rng = jax.random.PRNGKey(42)
weights = Weights.init(cfg, rng, mesh, rules)

In [ ]:
batch_size = 8
seq_len = 1024

input_sharding = _logical_to_sharding(("batch", "sequence"), mesh=mesh, rules=rules)
x = jnp.zeros((batch_size, seq_len), dtype=jnp.int32, device=input_sharding)

output_sharding = _logical_to_sharding(("batch", "sequence", "vocab"), mesh=mesh, rules=rules)
compiled = jax.jit(forward, out_shardings=output_sharding).lower(x, weights).compile()

In [ ]:
with jax.profiler.trace("/tmp/tensorboard"):
    logits = compiled(x, weights)
    jax.block_until_ready(logits)

In [ ]:
jax.tree.map(jnp.shape, weights.layers[0])

Layer(q=(8192, 16, 256), k=(8192, 16, 256), v=(8192, 16, 256), proj=(16, 256, 8192), w1=(8192, 32768), w2=(32768, 8192), gamma1=(8192,), gamma2=(8192,))